# ДЗ №3. Задание 2
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Частоты собственных имён.


## Подготовка

Используем Kafka и `telegram_producer.py` из задания 1.

Результаты каждого запуска хранятся отдельно в `output`. Checkpoint нужен Spark для накопительного подсчёта.

In [1]:
import json
import re
import socket
import sys
from datetime import datetime
from operator import add, itemgetter
from pathlib import Path
from threading import Event, Thread

from kafka import KafkaConsumer
from pyspark import SparkConf, SparkContext
from pyspark.streaming import StreamingContext

conf = SparkConf().setMaster("local[2]").setAppName("HW3-Names")
sc = SparkContext.getOrCreate(conf=conf)
sc.pythonExec = sys.executable
ssc = StreamingContext(sc, 60)

output = (Path("output") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")).resolve()
ssc.checkpoint(str(output / "checkpoint"))

## 1. Поток из Kafka

Читаем тот же топик `telegram_messages` и передаём JSON-строки в Spark.

In [2]:
consumer = KafkaConsumer(
    "telegram_messages", bootstrap_servers="localhost:9092",
    auto_offset_reset="latest", enable_auto_commit=False,
)
server = socket.socket()
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("127.0.0.1", 9998))
server.listen(1)
stopped = Event()


def forward_kafka():
    try:
        connection, _ = server.accept()
        with connection:
            while not stopped.is_set():
                records = consumer.poll(timeout_ms=1000)
                for batch in records.values():
                    for record in batch:
                        connection.sendall(record.value + b"\n")
    except (BrokenPipeError, ConnectionResetError):
        pass
    finally:
        consumer.close()
        server.close()


worker = Thread(target=forward_kafka, daemon=True)
lines = ssc.socketTextStream("127.0.0.1", 9998)

## 2. Отбор имён

По правилу из условия берём слова из русских букв, начинающиеся с заглавной.

In [3]:
def extract_names(line):
    text = json.loads(line)["text"]
    for word in re.findall(r"\b[А-ЯЁ][А-ЯЁа-яё]*\b", text):
        name = word.lower().rstrip("аеёиоуыэюяй")
        if name:
            yield name, 1

## 3. Накопление и сохранение каждую минуту

К предыдущему количеству добавляем новые вхождения. Сортируем по убыванию частоты и сохраняем весь результат в формате `слово, количество`.

In [4]:
def update_count(values, previous):
    return (previous or 0) + sum(values)


def format_count(item):
    word, count = item
    return f"{word}, {count}"


top_words = []


def save_counts(time, rdd):
    global top_words
    ordered = rdd.sortBy(itemgetter(1), ascending=False, numPartitions=1).cache()
    ordered.map(format_count).saveAsTextFile(str(output / time.strftime("%Y%m%d_%H%M%S")))
    top_words = ordered.take(10)
    ordered.unpersist()


minute_counts = lines.flatMap(extract_names).reduceByKey(add)
total_counts = minute_counts.updateStateByKey(update_count)
total_counts.foreachRDD(save_counts)

## 4. Запуск на 30 минут и итоговый топ-10

Если сборщик ещё не работает, после запуска этой ячейки запускаем `telegram_producer.py` в отдельном терминале. Поток работает 30 минут, завершает обработку принятых сообщений и выводит топ-10. Сборщик затем останавливаем через Ctrl+C.

Interrupt завершит сбор раньше срока. Для сдачи нужен полный запуск не менее 30 минут с подключённым сборщиком Telegram. Перед повторным запуском всех ячеек перезапускаем ядро.

In [5]:
worker.start()
ssc.start()
try:
    finish_at = datetime.now().timestamp() + 30 * 60
    while datetime.now().timestamp() < finish_at:
        if ssc.awaitTerminationOrTimeout(1):
            break
except KeyboardInterrupt:
    pass
finally:
    stopped.set()
    ssc.stop(stopSparkContext=False, stopGraceFully=True)

print("Слово, количество")
for word, count in top_words:
    print(f"{word}, {count}")

Слово, количество
тасс, 5
сайт, 5
в, 4
подписатьс, 4
макс, 4
владимир, 3
ирин, 3
мвд, 3
кмс, 3
рассылк, 3
